# 05 — Fixed-Target Reliability Summary

Run independently: this notebook loads current completed DB models and champion
results itself. It exports **the fixed-target attainment PNG** plus native and
noise terminal-reliability CSVs. It does not compute adaptive AUC profiles, load
cross-function traces, run evaluations, or display images inline.

Detailed strategy/convergence profiles are in `06_convergence_ecdf_and_ablations.ipynb`;
noise robustness is in `07_noise_robustness.ipynb`. Each export is
cached separately and refreshed only when relevant inputs, settings or code change.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from benchmark.infra.storage.analysis_results_store import AnalysisResultsStore
from shared.infra.lifecycle import WorkflowScope
from benchmark.application.analysis.analyze_reliability import AnalyzeReliability
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    noise_robustness_enabled=True, cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(), context={}, request_context={BenchmarkConfig: analysis_config})
reliability_analysis = services.get(AnalyzeReliability)
results_store = services.get(AnalysisResultsStore)


## Calculate reliability (no figure or report export)


In [2]:
reliability = reliability_analysis.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
reliability_snapshot = results_store.save(reliability)
print(f"Calculated {len(reliability.native)} current native condition rows.")
display(reliability.native)

print(f"Saved numerical results: {reliability_snapshot}")


Calculated 480 current native condition rows.


,Model,Strategy,Solver,Source Problem,Problem ID,Dim,Noise Std,Code Hash,Trials,Complete,Primary Successes,Primary Success Rate,Secondary Success Rate,CI Lower,CI Upper,Median Terminal Error
0,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,qwen_14b_baseline,1,1,10,0.0,cc1fc2823d31cf86d148520a6f4cb45e8268b2aaa58fa1...,20,True,0,0.00,0.00,0.000000,0.161125,4.539039e+01
1,qwen2.5-coder-14b-instruct-q4_k_m.gguf,baseline,qwen_14b_baseline_implicit,1,1,10,0.0,2fb4f19e3b5ff69ee21977585be3102fb0089f682ed5b2...,20,True,19,0.95,1.00,0.763869,0.991119,7.231904e-11
2,qwen2.5-coder-14b-instruct-q4_k_m.gguf,guided,qwen_14b_guided,1,1,10,0.0,19784f919d5fe235a60006c48426fc79f0d62ae9b1aa98...,20,True,0,0.00,1.00,0.000000,0.161125,4.425477e-04
3,qwen2.5-coder-14b-instruct-q4_k_m.gguf,guided,qwen_14b_guided_implicit,1,1,10,0.0,5ab1bc653b12cae6adec8050d767fce680406f848c6eff...,20,True,0,0.00,0.35,0.000000,0.161125,1.683226e+01
4,qwen2.5-coder-14b-instruct-q4_k_m.gguf,thinking,qwen_14b_thinking,1,1,10,0.0,f19c9c2a4918a6348a1ef0ef0262d2433272be3107a681...,20,True,20,1.00,1.00,0.838875,1.000000,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,qwen2.5-coder-14b-instruct-q4_k_m.gguf,guided,qwen_14b_guided_noisy,8,8,50,0.2,6472a27e26ab13e72a8b4f62f86d927309fe444b964a74...,20,True,0,0.00,0.00,0.000000,0.161125,1.840614e+04
476,qwen2.5-coder-14b-instruct-q4_k_m.gguf,thinking,qwen_14b_thinking_implicit,8,8,50,0.2,12085d1c504bcc70f8d7373a49869c7975ca9b8762f64b...,20,True,0,0.00,0.00,0.000000,0.161125,3.537412e+02
477,qwen2.5-coder-14b-instruct-q4_k_m.gguf,thinking,qwen_14b_thinking_noisy,8,8,50,0.2,153b80b50d4d1dbdef336876c186b47718902eb9c340a9...,20,True,0,0.00,0.00,0.000000,0.161125,6.309091e+01
478,qwen2.5-coder-14b-instruct-q4_k_m.gguf,vectorization,qwen_14b_vectorization_implicit,8,8,50,0.2,7f77cbfa16997c58786f4c81221a37c5498d8f980effc8...,20,True,0,0.00,0.00,0.000000,0.161125,9.000107e+02


Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/reliability/4c260af41d543de542e0d7e59c5e0401fdbaa4ad94a20b8abe0643fcad128594


## Reliability tables and fixed-target attainment export

The matrix scores returned solutions. The attainment curve describes the best
clean point queried, which need not be returned. Noise reliability uses identical
frozen champions, not separately synthesized noisy champions. Incomplete conditions
are excluded from aggregate figures, not counted as failures.


In [3]:
figure_exporter.export_reliability(reliability)
